In [3]:
import torch
import torch.nn as nn

from abc import ABC, abstractmethod

### Base Classes required to sample from an SDE for a density diffusion process

In [5]:
class SDE(ABC):
    @abstractmethod
    def drift(self, xt: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        pass

    @abstractmethod
    def diffusion(self, xt: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        pass

class Simulator(ABC):
    @abstractmethod
    def step(self, xt:torch.Tensor, t:torch.Tensor, h: torch.Tensor) -> torch.Tensor:
        pass

    @torch.no_grad()
    def simulate(self, x:torch.Tensor, ts) -> torch.Tensor:
        for ts_idx in range(len(ts)+1):
            t = ts[ts_idx]
            h = ts[ts_idx] - ts[ts_idx-1]
            x = self.step(x,t,h)
        return x

    @torch.no_grad()
    def simulate_with_trajectory(self, x:torch.Tensor, ts) -> torch.Tensor:
        xs = [x.clone()]
        for ts_idx in range(len(ts)+1):
            t = ts[ts_idx]
            h = ts[ts_idx] - ts[ts_idx-1]
            x = self.step(x,t,h)
            xs.append(x)
        return torch.stack(xs, dim=1)

class Density(ABC):
    @abstractmethod
    def log_density(self, xt: torch.Tensor) -> torch.Tensor:
        pass

    @abstractmethod
    def score(self, xt: torch.Tensor) -> torch.Tensor:
        pass

class Samplable(ABC):
    @abstractmethod
    def sample(self):
        pass

In [6]:
class EulerMaruyamaSimulator(Simulator):
    def __init__(self, sde: SDE):
        self.sde = sde

    def step(self, xt: torch.Tensor, t: torch.Tensor, h:torch.Tensor) -> torch.Tensor:
        return xt + h * self.sde.drift(xt, t) + torch.sqrt(h) * self.sde.diffusion(xt, t) * torch.randn_like(xt)